In [1]:
import pandas as pd

df_2024 = pd.read_csv("../data/raw/atp_matches_2024.csv")
df_2025 = pd.read_csv("../data/raw/atp_matches_2025.csv")

df = pd.concat([df_2024, df_2025])

print("Matches loaded:", len(df))

Matches loaded: 6020


In [2]:
# Feature Engineering

df["rank_difference"] = (
    df["loser_rank"] -
    df["winner_rank"]
)

df["rank_points_difference"] = (
    df["winner_rank_points"] -
    df["loser_rank_points"]
)

df["age_difference"] = (
    df["winner_age"] -
    df["loser_age"]
)

print("Features created")

Features created


In [3]:
v1_df = df[
    [
        "surface",
        "tourney_level",
        "round",
        "best_of",
        "rank_difference",
        "rank_points_difference",
        "age_difference"
    ]
].copy()

print(v1_df.head())

  surface tourney_level round  best_of  rank_difference  \
0    Hard             A     F        3             -6.0   
1    Hard             A    SF        3             31.0   
2    Hard             A    SF        3             41.0   
3    Hard             A    QF        3            108.0   
4    Hard             A    QF        3              5.0   

   rank_points_difference  age_difference  
0                 -1090.0            12.0  
1                  2538.0            -5.8  
2                  1668.0             2.9  
3                  3087.0           -11.3  
4                   101.0             3.6  


In [4]:
v1_df["target"] = 1

print(v1_df.head())

  surface tourney_level round  best_of  rank_difference  \
0    Hard             A     F        3             -6.0   
1    Hard             A    SF        3             31.0   
2    Hard             A    SF        3             41.0   
3    Hard             A    QF        3            108.0   
4    Hard             A    QF        3              5.0   

   rank_points_difference  age_difference  target  
0                 -1090.0            12.0       1  
1                  2538.0            -5.8       1  
2                  1668.0             2.9       1  
3                  3087.0           -11.3       1  
4                   101.0             3.6       1  


In [5]:
print(len(v1_df))

6020


In [6]:
wins_df = v1_df.copy()

wins_df["target"] = 1

print(len(wins_df))
wins_df.head()

6020


,surface,tourney_level,round,best_of,rank_difference,rank_points_difference,age_difference,target
0,Hard,A,F,3,-6.0,-1090.0,12.0,1
1,Hard,A,SF,3,31.0,2538.0,-5.8,1
2,Hard,A,SF,3,41.0,1668.0,2.9,1
3,Hard,A,QF,3,108.0,3087.0,-11.3,1
4,Hard,A,QF,3,5.0,101.0,3.6,1


In [7]:
losses_df = v1_df.copy()

losses_df["rank_difference"] *= -1
losses_df["rank_points_difference"] *= -1
losses_df["age_difference"] *= -1

losses_df["target"] = 0

print(len(losses_df))
losses_df.head()

6020


,surface,tourney_level,round,best_of,rank_difference,rank_points_difference,age_difference,target
0,Hard,A,F,3,6.0,1090.0,-12.0,0
1,Hard,A,SF,3,-31.0,-2538.0,5.8,0
2,Hard,A,SF,3,-41.0,-1668.0,-2.9,0
3,Hard,A,QF,3,-108.0,-3087.0,11.3,0
4,Hard,A,QF,3,-5.0,-101.0,-3.6,0


In [8]:
training_df = pd.concat(
    [wins_df, losses_df],
    ignore_index=True
)

print(training_df.shape)

training_df.head()

(12040, 8)


,surface,tourney_level,round,best_of,rank_difference,rank_points_difference,age_difference,target
0,Hard,A,F,3,-6.0,-1090.0,12.0,1
1,Hard,A,SF,3,31.0,2538.0,-5.8,1
2,Hard,A,SF,3,41.0,1668.0,2.9,1
3,Hard,A,QF,3,108.0,3087.0,-11.3,1
4,Hard,A,QF,3,5.0,101.0,3.6,1


In [9]:
print(training_df["target"].value_counts())

target
1    6020
0    6020
Name: count, dtype: int64


In [10]:
print(training_df.shape)

(12040, 8)


In [11]:
print(training_df["surface"].unique())
print(training_df["tourney_level"].unique())
print(training_df["round"].unique())

<StringArray>
['Hard', 'Clay', 'Grass']
Length: 3, dtype: str
<StringArray>
['A', 'G', 'M', 'O', 'F', 'D']
Length: 6, dtype: str
<StringArray>
['F', 'SF', 'QF', 'R16', 'R32', 'RR', 'R128', 'R64', 'BR']
Length: 9, dtype: str
The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [12]:
training_encoded = training_df.copy()

# Surface
surface_map = {
    "Hard": 0,
    "Clay": 1,
    "Grass": 2
}

training_encoded["surface"] = (
    training_encoded["surface"]
    .map(surface_map)
)

# Tournament Level
level_map = {
    "A": 0,
    "M": 1,
    "G": 2,
    "D": 3,
    "F": 4,
    "O": 5
}

training_encoded["tourney_level"] = (
    training_encoded["tourney_level"]
    .map(level_map)
)

# Round
round_map = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7,
    "RR": 8,
    "BR": 9
}

training_encoded["round"] = (
    training_encoded["round"]
    .map(round_map)
)

print(training_encoded.head())

   surface  tourney_level  round  best_of  rank_difference  \
0        0              0      7        3             -6.0   
1        0              0      6        3             31.0   
2        0              0      6        3             41.0   
3        0              0      5        3            108.0   
4        0              0      5        3              5.0   

   rank_points_difference  age_difference  target  
0                 -1090.0            12.0       1  
1                  2538.0            -5.8       1  
2                  1668.0             2.9       1  
3                  3087.0           -11.3       1  
4                   101.0             3.6       1  


In [13]:
print(training_encoded.isnull().sum())

surface                     0
tourney_level               0
round                       0
best_of                     0
rank_difference           210
rank_points_difference    210
age_difference             20
target                      0
dtype: int64


In [14]:
training_clean = training_encoded.dropna()

print("Before:", len(training_encoded))
print("After :", len(training_clean))
print("Rows Removed:", len(training_encoded) - len(training_clean))

print("\n")

print(training_clean.isnull().sum())

Before: 12040
After : 11828
Rows Removed: 212


surface                   0
tourney_level             0
round                     0
best_of                   0
rank_difference           0
rank_points_difference    0
age_difference            0
target                    0
dtype: int64
